In [1]:
import numpy as np
import pandas as pd
import os
import warnings
import matplotlib.pyplot as plt
from arch import arch_model
from arch.univariate import ARX
import scipy
from scipy.stats import norm
from sklearn.mixture import GaussianMixture
from scipy.stats import kurtosis, skew
from scipy.optimize import minimize

### Import data

In [2]:
path = os.path.abspath('E:/RA/Geert/task1.py')
dir_path = os.path.dirname(path)
os.chdir(dir_path)
excel_file = pd.ExcelFile('Aggregate_CPI_inflation_20230513.xls')
sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
#quarterly and monthly aggregate CPI data (deseasonalized). The full sample is 1947-2022
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))
data_month.index = pd.to_datetime(data_month[['Year', 'Month']].assign(day=1))
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
sample_data = data_quarter[data_quarter['Year']>1969]

In [5]:
sample_data['Inflation_lag_1'] =  sample_data.loc[:,'Inflation'].shift(1)
sample_data['Inflation_lag_2'] =  sample_data.loc[:,'Inflation'].shift(2)
sample_data['Forecasted_inflation_lag_1'] =  sample_data.loc[:,'Forecasted inflation'].shift(1)
sample_data = sample_data.dropna()

 ### Gaussian Mixture distribution

The mixture of 2 normal distribution given parameters $p_1$,$\mu_1$,$\sigma^2_1$ is 
$$
 f\left(z_t\right) =   
 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z_t-\mu_1)^2}{2\sigma_1^2} \}
+p_2\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z_t-\mu_2)^2}{2\sigma_2^2} \}
$$
where $p_2 = 1- p_1 $,$\mu_2 = \frac{-p_1\mu_1}{p_2}$, and
$\sigma_2^2 =\frac{1-p_2\mu_2^2 -p_1(\sigma_1^2 + \mu_1^2 )}{p_2} $

N th order moment can be calculated as:
$$
E[z^n] = \int z^n   \left[p_1 \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z-\mu_1)^2}{2\sigma_1^2} \}
+(1-p_1)\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z-\mu_2)^2}{2\sigma_2^2} \}  \right] dz
$$

$$
E[z^n] =  p_1 E[x_1^n|x_1 \sim N(\mu_1,\sigma_1^2) ] + p_2 E[x_2^n|x_2 \sim N(\mu_2,\sigma_2^2) ]
$$

Similary, CDF can be calculated analytically by
$$
\Phi(a) = \int_{-\inf}^{a} z   \left[p_1 \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z-\mu_1)^2}{2\sigma_1^2} \}
+p_2\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z-\mu_2)^2}{2\sigma_2^2} \}  \right] dz
$$

$$
\Phi(a) = p_1\Phi_{x_1}(a) + p_2\Phi_{x_2}(a)
$$

We can think mixture of two normal distributions generated by three random variable $Z$,$X_1$,$X_2$. $Z$ follows Bernoulli($P_1$), $X_1 \sim  N(\mu_1,\sigma_1^2)$ and $X_2 \sim  N(\mu_2,\sigma_2^2)$. In order to generate a random sample point, we can first use $Z$ to decide which normal distributions to use, and then randomly pick a point in that normal distribution. 

Log likelihood function is
$$
 L\left(\epsilon_t\right) =  \sum_{t=1}^{T} \ln \frac{1}{\sqrt{h_t}} \left[
 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z_t-\mu_1)^2}{2\sigma_1^2} \}
+p_2\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z_t-\mu_2)^2}{2\sigma_2^2} \} \right]
$$,
where $z_t = \frac{\epsilon_t}{\sqrt{h_t}}$

In [190]:
from __future__ import annotations
from arch.univariate.distribution import Distribution
from abc import ABCMeta, abstractmethod
from collections.abc import Sequence
from typing import Callable
import warnings

from numpy import (
    abs,    array,    asarray,    empty,    exp,    int64,    
    integer,    isscalar,    log,    nan,    ndarray, exp,
    ones_like,    pi,    sign,    sqrt,    sum, random)
from numpy.random import Generator, RandomState, default_rng
from scipy.special import comb, gamma, gammainc, gammaincc, gammaln
import scipy.stats as stats
from scipy.optimize import bisect

from arch.typing import ArrayLike, ArrayLike1D, Float64Array
from arch.utility.array import AbstractDocStringInheritor, ensure1d

class MixNormal(Distribution, metaclass=AbstractDocStringInheritor):
    """
    Mixture of two Normal distributions for use with SPARCH model

    Parameters
    ----------
    random_state : RandomState, optional
        .. deprecated:: 5.0

           random_state is deprecated. Use seed instead.

    seed : {int, Generator, RandomState}, optional
        Random number generator instance or int to use. Set to ensure
        reproducibility. If using an int, the argument is passed to
        ``np.random.default_rng``.  If not provided, ``default_rng``
        is used with system-provided entropy.
    """

    def __init__(
        self,
        random_state: RandomState | None = None,
        *,
        seed: None | int | RandomState | Generator = None,
    ) -> None:
        super().__init__(random_state=random_state, seed=seed)
        self._name = "Mixture of two Normal distributions"
        self.num_params: int = 3  

    def constraints(self) -> tuple[Float64Array, Float64Array]:
        return empty(0), empty(0)
        #return array([[1, 0, 0], [-1, 0, 0], [0, 1,0], [0, -1,0] , [0,0,1],[0,0,-1]]), array([0.05, 0.95, -20,20,0.2, 10])

    def bounds(self, resids: Float64Array) -> list[tuple[float, float]]:
        """
        Bounds of parameters:
        p1: (0,1)
        u1: (-10,10)
        sigma_1:(0.001,10)
        """
        return [(0.0001, 0.9999),(-10000,10000),(0.0001, 10000)]

    def loglikelihood(
        self,
        parameters: Sequence[float] | ArrayLike1D,
        resids: ArrayLike,
        sigma2: ArrayLike,
        individual: bool = False,
    ) -> float | Float64Array:
        r"""Computes the log-likelihood of assuming residuals are mixture normally
        distributed, conditional on the variance

        Parameters
        ----------
        parameters : ndarray
            Parameters of the first normal distribution: p1,u1,sigma1. Second one can be calculated by restrictions.
        resids  : ndarray
            The residuals to use in the log-likelihood calculation
        sigma2 : ndarray
            Conditional variances of resids
        individual : bool, optional
            Flag indicating whether to return the vector of individual log
            likelihoods (True) or the sum (False)

        Returns
        -------
        ll : float
            The log-likelihood

        Notes
        -----
        The log-likelihood of a single data point x is

        .. math::

            \ln f\left(x\right)=
            \ln \frac{1}{\sqrt{h_t}} \left[
                 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(x-\mu_1)^2}{2\sigma_1^2} \}
                +(1-p_1)\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(x-\mu_2)^2}{2\sigma_2^2} \} \right]

        """
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        z = resids/sqrt(sigma2)
        warnings.filterwarnings("ignore")
        lls =-1/2 *log(sigma2) +  log(   p1/(sqrt(2*pi*sigma_1_2)) *  exp(-( z-u1)**2/(2*sigma_1_2)) 
                                     +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
        warnings.filterwarnings("default")
          
        if individual:
            return lls
        else:
            return sum(lls)

    def starting_values(self, std_resid: Float64Array) -> Float64Array:
        """
        Starting values of parameters
        """
        #gmm = GaussianMixture(n_components=2).fit(std_resid.reshape(-1,1))
        #return array([gmm.weights_[0],gmm.means_[0][0], gmm.covariances_[0][0][0] ])
        return array([0.3, 0.1,1])
    
    def _simulator(self, size: int | tuple[int, ...]) -> Float64Array:
        assert self._parameters is not None
        p1, u1, sigma_1_2 = self._parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        Z = np.random.binomial(n=1, p=p1, size=size)
        return sqrt(sigma_1_2)**Z*sqrt(sigma_2_2)**(1-Z)*self._generator.standard_normal(size) + u1*Z+ u2*(1-Z)

    def simulate(
        self, parameters: int | float | Sequence[float | int] | ArrayLike1D
    ) -> Callable[[int | tuple[int, ...]], Float64Array]:
        parameters = ensure1d(parameters, "parameters", False)
        self._parameters = asarray(parameters, dtype=float)
        return self._simulator

    def parameter_names(self) -> list[str]:
        return ['p_1','mu_1','sigma_1^2']

    def cdf(
        self,
        resids: Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        return p1*stats.norm.cdf(asarray((resids-u1)/sqrt(sigma_1_2))  ) + p2*stats.norm.cdf(asarray((resids-u2)/sqrt(sigma_2_2)))

    def ppf(
        self,
        pits: float | Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        scalar = isscalar(pits)
        if scalar:
            pits = array([pits])
        else:
            pits = asarray(pits)
            
        def inverse_cdf(cdf, target_p, lower_bound=-100, upper_bound=100):
            def root_func(x):
                return cdf(x,parameters) - target_p
            return bisect(root_func, lower_bound, upper_bound)     
        
        ppf = inverse_cdf(self.cdf, pits, lower_bound=-100, upper_bound=100)

        if scalar:
            return ppf[0]
        else:
            return ppf

    def moment(
        self, n: int, parameters: None | Sequence[float] | ArrayLike1D = None
    ) -> float:
        if n < 0:
            return nan
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        moment1 = stats.norm.moment(n,loc=u1,scale=sqrt(sigma_1_2))
        moment2 = stats.norm.moment(n,loc=u2,scale=sqrt(sigma_2_2))
        return p1 * moment1 + p2 * moment2

    def partial_moment(
        self,
        n: int,
        z: float = 0.0,
        parameters: None | Sequence[float] | ArrayLike1D = None,
        num_samples=1000,
    ) -> float:
        
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        if n < 0:
            return nan
        elif n == 0:
            return cdf(z,parameters)
        elif n==1:
            return -p1*stats.norm.pdf(z,loc=u1,scale=sqrt(sigma_1_2))  -p2*stats.norm.pdf(z,loc=u2,scale=sqrt(sigma_2_2))
        else:
            -(z ** (n - 1)) * (p1*stats.norm.pdf(z,loc=u1,scale=sqrt(sigma_1_2))+p2*stats.norm.pdf(z,loc=u2,scale=sqrt(sigma_2_2))) 
            + (n - 1) * self.partial_moment(  n - 2, z, parameters  )

In [27]:
def print_param(params):
    """print distributional parameters"""
    p1,u1,sigma_1_2 = params
    p2 = 1- p1
    u2 = p1/(p1-1) *u1
    sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
    print('Mixture of 2 normals parameters: ')
    print("Probabilities: ", [p1,p2])
    print("Means: ", [u1,u2])
    print("Variances: ", [sigma_1_2,sigma_2_2])

In [286]:
def MofN_estimation(errors,volatility):
    z = errors / volatility
    def MofN_llf(params):
        p1,u1,sigma_1_2 = params
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        lls =-np.log(volatility) +   np.log(p1 * norm.pdf(z, u1, sigma_1_2**0.5) + p2 * norm.pdf(z, u2, sigma_2_2**0.5))
        return -np.sum(lls)
    init_params = [0.3, 0.1,1]
    # Bound constraints, sigma > 0, 0 < p1 < 1
    bnds = ((1e-5, 1 - 1e-5), (None, None), (1e-5, None))
    # Constraints 
    constraints = {'type': 'ineq',
                    'fun': lambda x: np.array([ 1 - 1e-5 - x[0] * (x[1]**2 + x[2]) - (1 - x[0]) * (x[0] * x[1] / (1 - x[0]))**2])} 
    res = minimize(MofN_llf, init_params, bounds=bnds, constraints=constraints)
    print_param(res.x)
    print('Whether converge:',res.success,'.  ', res.message)

In [315]:
def SPARCH(Y,dparams,X=None,mean='Zero', vol='GARCH',p=1,o=0,q=1,lags=None,cov = 'robust'):
    '''
    step 1: run a garch with student t distribution; estimate gaussian mixture parameters from garch residuals
    step 2: run a sparch using garch parameters
    '''
    #run GARCH with student t distribuion as banchmark
    garht = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=p, o=o,q=q,lags=lags,dist='studentst').fit(disp='off',cov_type=cov)
    print('\033[1m Using residuals of t distribution to estimate mixture of 2 normals parameters \033[0m')
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        MofN_estimation(garht.resid,garht.conditional_volatility)
    #residuals = (garht.resid/garht.conditional_volatility).dropna()
    #print('Distribution Property of student t Residuals')
    #print('Kurtosis: ',kurtosis(residuals, fisher=True),'Skewness: ',skew(residuals))
    params = array(garht.params) +   0.1*array(garht.std_err)*np.random.normal(size=garht.std_err.shape)
    starting_values = np.concatenate( (np.array(params)[:-1],dparams )   )
    constraints = {'type': 'ineq',
                'fun': lambda x: array([ 0.9999 - x[-3] * (x[-2]**2 + x[-1]) - (1 - x[-3]) * (-x[-3] * x[-2] / (1 - x[-3]))**2])} 
    options={'maxiter': 1000,'constraints':constraints}
    sparch = arch_model(y=Y,x=X, mean=mean, vol='GARCH',p=p, o=o,q=q,lags=lags)
    sparch.distribution = MixNormal()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        MixNormalres=sparch.fit(disp='off',cov_type=cov,options=options,starting_values=starting_values )
    print('\n\033[1m SPARCH estimation result: mixture of 2 normals parameters \033[0m')
    print_param(MixNormalres.params[-3:])
    return MixNormalres

$
\pi (t) =  fc(t-1) + \epsilon (t)
$

In [299]:
SPARCH(Y = sample_data['Inflation shock'],dparams=[0.34,0.1,1],mean='Zero', vol='GARCH',p=1,o=1,q=1,lags=None,cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.0839235947822457, 0.9160764052177544]
Means:  [-1.0430979474836386, 0.09556029274872595]
Variances:  [5.228113220988753, 0.6026857278182806]
Whether converge: True .   Optimization terminated successfully

 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.08809186356779938, 0.9119081364322006]
Means:  [-1.0965166590510373, 0.10592535812528618]
Variances:  [5.68905550566661, 0.5347252618876738]


                              Zero Mean - GJR-GARCH Model Results                              
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -168.174
Distribution:      Mixture of two Normal distributions   AIC:                           350.348
Method:                             Maximum Likelihood   BIC:                           373.711
                                                         No. Observations:                  208
Date:                                 Tue, Jul 25 2023   Df Residuals:                      208
Time:                                         20:38:35   Df Model:                            0
                              Volatility Model                             
                 coef    std err          t      P>|t|     9

In [308]:
SPARCH(Y = sample_data['Inflation shock'],dparams=[0.34,0.1,1],mean='Zero', vol='GARCH',p=2,o=2,q=1,lags=None,cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.08142246267594931, 0.9185775373240507]
Means:  [-0.8118815550439428, 0.07196495987201222]
Variances:  [4.858074652114801, 0.6523832971766015]
Whether converge: True .   Optimization terminated successfully

 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.075453538558283, 0.924546461441717]
Means:  [-1.1403305279973204, 0.09306398007219789]
Variances:  [6.18727091506987, 0.5672917835134861]


                              Zero Mean - GJR-GARCH Model Results                              
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -164.833
Distribution:      Mixture of two Normal distributions   AIC:                           347.666
Method:                             Maximum Likelihood   BIC:                           377.704
                                                         No. Observations:                  208
Date:                                 Tue, Jul 25 2023   Df Residuals:                      208
Time:                                         20:40:49   Df Model:                            0
                              Volatility Model                             
                 coef    std err          t      P>|t|     9

In [320]:
SPARCH(Y = sample_data['Inflation shock'],dparams=[0.34,0.1,1],mean='Zero', vol='GARCH',p=1,o=1,q=2,lags=None,cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.08393107002753962, 0.9160689299724604]
Means:  [-1.0431385065400465, 0.09557330041033897]
Variances:  [5.227837698129494, 0.602670513376056]
Whether converge: True .   Optimization terminated successfully

 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.08806279898197045, 0.9119372010180296]
Means:  [-1.0978976313935793, 0.10602039077719647]
Variances:  [5.690969431848035, 0.5346826284811671]


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "


                              Zero Mean - GJR-GARCH Model Results                              
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -168.174
Distribution:      Mixture of two Normal distributions   AIC:                           352.348
Method:                             Maximum Likelihood   BIC:                           379.048
                                                         No. Observations:                  208
Date:                                 Tue, Jul 25 2023   Df Residuals:                      208
Time:                                         20:46:12   Df Model:                            0
                              Volatility Model                             
                 coef    std err          t      P>|t|     9

In [317]:
SPARCH(Y = sample_data['Inflation shock'],dparams=[0.34,0.1,1],mean='Zero', vol='GARCH',p=2,o=2,q=2,lags=None,cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.08141269862045704, 0.9185873013795429]
Means:  [-0.8119973295821185, 0.07196582597495632]
Variances:  [4.858053591667246, 0.6524297321019495]
Whether converge: True .   Optimization terminated successfully

 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.8978234941845302, 0.10217650581546978]
Means:  [0.05359592757367915, -0.4709466484904613]
Variances:  [0.48485192296223234, 3.3559359258069894]


D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)


                              Zero Mean - GJR-GARCH Model Results                              
Dep. Variable:                         Inflation shock   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.005
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -165.181
Distribution:      Mixture of two Normal distributions   AIC:                           350.363
Method:                             Maximum Likelihood   BIC:                           383.738
                                                         No. Observations:                  208
Date:                                 Tue, Jul 25 2023   Df Residuals:                      208
Time:                                         20:45:53   Df Model:                            0
                             Volatility Model                             
                 coef    std err          t      P>|t|    95.

$
\pi (t) = c +\rho \pi (t-1)   + \phi fc(t-1) + \epsilon (t)
$

In [269]:
SPARCH(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=1,o=1,q=1,lags=[1],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.057854631520000296, 0.9421453684799997]
Means:  [-0.46984413181910106, 0.028851873636107125]
Variances:  [6.492501805495073, 0.66183658132696]
Whether converge: True .   Optimization terminated successfully


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "



 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.05973201152317612, 0.9402679884768239]
Means:  [-0.5737426402060578, 0.036447908911205536]
Variances:  [6.741383131747998, 0.6338567114187939]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.437
Mean Model:                                       AR-X   Adj. R-squared:                  0.432
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -164.110
Distribution:      Mixture of two Normal distributions   AIC:                           348.221
Method:                             Maximum Likelihood   BIC:                           381.548
                                                         No. Observations:                  207
Date:                                 Tue, Jul 25 2023   Df Residuals:                      204
Time:                                         20:12:55   Df Model:                            3
                                      Mean Model                                     
                           coef    std err        

In [270]:
SPARCH(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=2,o=2,q=1,lags=[1],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.07485684431354919, 0.9251431556864508]
Means:  [-0.23996764315883815, 0.019416692858633185]
Variances:  [5.0032083352640315, 0.6756777054179282]
Whether converge: True .   Optimization terminated successfully


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outs


 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.9293729075877074, 0.07062709241229259]
Means:  [0.02362010588455439, -0.31081396293806823]
Variances:  [0.6049106816244991, 4.831107578925671]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.437
Mean Model:                                       AR-X   Adj. R-squared:                  0.432
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -160.908
Distribution:      Mixture of two Normal distributions   AIC:                           345.816
Method:                             Maximum Likelihood   BIC:                           385.809
                                                         No. Observations:                  207
Date:                                 Tue, Jul 25 2023   Df Residuals:                      204
Time:                                         20:13:28   Df Model:                            3
                                      Mean Model                                     
                           coef    std err        

In [271]:
SPARCH(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=1,o=1,q=2,lags=[1],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.057855283210658665, 0.9421447167893413]
Means:  [-0.4698117553769079, 0.02885023041434222]
Variances:  [6.492496509178405, 0.6618329740756067]
Whether converge: True .   Optimization terminated successfully

 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.9406102037050664, 0.05938979629493357]
Means:  [0.024363664449100122, -0.3858695063822762]
Variances:  [0.5584473120804565, 5.486186254145291]


D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.439
Mean Model:                                       AR-X   Adj. R-squared:                  0.433
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -164.104
Distribution:      Mixture of two Normal distributions   AIC:                           350.207
Method:                             Maximum Likelihood   BIC:                           386.867
                                                         No. Observations:                  207
Date:                                 Tue, Jul 25 2023   Df Residuals:                      204
Time:                                         20:13:28   Df Model:                            3
                                      Mean Model                                     
                           coef    std err        

In [272]:
SPARCH(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=2,o=2,q=2,lags=[1],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.07487405574681429, 0.9251259442531857]
Means:  [-0.24003621210635354, 0.01942706810694155]
Variances:  [5.002861703892098, 0.6756248128123953]
Whether converge: True .   Optimization terminated successfully


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outs


 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.9294039005566274, 0.07059609944337264]
Means:  [0.023622614329274955, -0.31099380946087224]
Variances:  [0.604849616107067, 4.832182807643101]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.437
Mean Model:                                       AR-X   Adj. R-squared:                  0.432
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -160.908
Distribution:      Mixture of two Normal distributions   AIC:                           347.816
Method:                             Maximum Likelihood   BIC:                           391.141
                                                         No. Observations:                  207
Date:                                 Tue, Jul 25 2023   Df Residuals:                      204
Time:                                         20:13:29   Df Model:                            3
                                      Mean Model                                     
                           coef    std err        

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi fc(t-1) + \epsilon (t)
$

In [273]:
SPARCH(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=1,o=1,q=1,lags=[1,2],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.05368465957167556, 0.9463153404283244]
Means:  [-0.49539529326226167, 0.028103874613463753]
Variances:  [6.989696493175464, 0.6593686785250968]
Whether converge: True .   Optimization terminated successfully


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bo


 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.04993055400858436, 0.9500694459914156]
Means:  [-0.6639550954318015, 0.03489391843049744]
Variances:  [8.070919153126592, 0.6271088416707633]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.444
Mean Model:                                       AR-X   Adj. R-squared:                  0.435
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -163.015
Distribution:      Mixture of two Normal distributions   AIC:                           348.030
Method:                             Maximum Likelihood   BIC:                           384.637
                                                         No. Observations:                  206
Date:                                 Tue, Jul 25 2023   Df Residuals:                      202
Time:                                         20:16:22   Df Model:                            4
                                      Mean Model                                     
                           coef    std err        

In [323]:
SPARCH(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=2,o=2,q=1,lags=[1,2],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.07529771760948101, 0.924702282390519]
Means:  [-0.2540518411617622, 0.020687224589209286]
Variances:  [5.009483909185982, 0.6730483376408846]
Whether converge: True .   Optimization terminated successfully

 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.07124954826791213, 0.9287504517320879]
Means:  [-0.35094147555817323, 0.02692264811862215]
Variances:  [5.359984174638307, 0.6647412323738345]


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.443
Mean Model:                                       AR-X   Adj. R-squared:                  0.435
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -160.025
Distribution:      Mixture of two Normal distributions   AIC:                           346.051
Method:                             Maximum Likelihood   BIC:                           389.313
                                                         No. Observations:                  206
Date:                                 Tue, Jul 25 2023   Df Residuals:                      202
Time:                                         20:46:32   Df Model:                            4
                                      Mean Model                                     
                           coef    std err        

In [275]:
SPARCH(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=1,o=1,q=2,lags=[1,2],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.05369356955813095, 0.946306430441869]
Means:  [-0.49552664313997774, 0.028116256452912217]
Variances:  [6.988878935836552, 0.659354727788365]
Whether converge: True .   Optimization terminated successfully


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outs


 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.9535236856274788, 0.0464763143725212]
Means:  [0.016785025439947435, -0.344367223109939]
Variances:  [0.591109245486097, 6.837344001422159]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.446
Mean Model:                                       AR-X   Adj. R-squared:                  0.437
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -163.051
Distribution:      Mixture of two Normal distributions   AIC:                           350.102
Method:                             Maximum Likelihood   BIC:                           390.036
                                                         No. Observations:                  206
Date:                                 Tue, Jul 25 2023   Df Residuals:                      202
Time:                                         20:16:24   Df Model:                            4
                                      Mean Model                                     
                           coef    std err        

In [276]:
SPARCH(Y = sample_data['Inflation'],X=sample_data['Forecasted inflation'],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=2,o=2,q=2,lags=[1,2],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.07528524031693538, 0.9247147596830646]
Means:  [-0.254228820643401, 0.020697926206119405]
Variances:  [5.009923892450195, 0.6730705497243459]
Whether converge: True .   Optimization terminated successfully


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outs


 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.9291819279877677, 0.07081807201223234]
Means:  [0.02647848615932886, -0.34741599313057975]
Variances:  [0.5847406833878565, 4.74415135300277]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.443
Mean Model:                                       AR-X   Adj. R-squared:                  0.435
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -160.019
Distribution:      Mixture of two Normal distributions   AIC:                           348.038
Method:                             Maximum Likelihood   BIC:                           394.628
                                                         No. Observations:                  206
Date:                                 Tue, Jul 25 2023   Df Residuals:                      202
Time:                                         20:16:24   Df Model:                            4
                                      Mean Model                                     
                           coef    std err        

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi_1 fc(t-1) + \phi_2 fc(t-2) + \epsilon (t)
$

In [277]:
SPARCH(Y = sample_data['Inflation'],X=sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=1,o=1,q=1,lags=[1,2],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.05335045774470558, 0.9466495422552944]
Means:  [-0.4942896902782166, 0.027856751688704592]
Variances:  [7.025552516089867, 0.6595973809430101]
Whether converge: True .   Optimization terminated successfully


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "



 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.04997127087557638, 0.9500287291244236]
Means:  [-0.6816839211688266, 0.035856401845497356]
Variances:  [8.147430502726705, 0.6226936551700079]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.443
Mean Model:                                       AR-X   Adj. R-squared:                  0.431
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -162.948
Distribution:      Mixture of two Normal distributions   AIC:                           349.895
Method:                             Maximum Likelihood   BIC:                           389.830
                                                         No. Observations:                  206
Date:                                 Tue, Jul 25 2023   Df Residuals:                      201
Time:                                         20:16:26   Df Model:                            5
                                         Mean Model                                        
                                 coef    std

In [288]:
SPARCH(Y = sample_data['Inflation'],X=sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=2,o=2,q=1,lags=[1,2],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 


C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_10388/1911162260.py:8: RuntimeWarning: invalid value encountered in double_scalars
  lls =-np.log(volatility) +   np.log(p1 * norm.pdf(z, u1, sigma_1_2**0.5) + p2 * norm.pdf(z, u2, sigma_2_2**0.5))
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: Runti

Mixture of 2 normals parameters: 
Probabilities:  [0.03294892319045908, 0.9670510768095409]
Means:  [-2.284069103060171, 0.0778217606527232]
Variances:  [6.299010074593635, 0.8131919743470556]
Whether converge: True .   Optimization terminated successfully

 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.9267712496600504, 0.07322875033994958]
Means:  [0.028450566267817275, -0.3600657765038579]
Variances:  [0.5718597607018957, 4.648030315341783]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.440
Mean Model:                                       AR-X   Adj. R-squared:                  0.429
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -159.903
Distribution:      Mixture of two Normal distributions   AIC:                           347.807
Method:                             Maximum Likelihood   BIC:                           394.397
                                                         No. Observations:                  206
Date:                                 Tue, Jul 25 2023   Df Residuals:                      201
Time:                                         20:25:48   Df Model:                            5
                                         Mean Model                                        
                                 coef    std

In [279]:
SPARCH(Y = sample_data['Inflation'],X=sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']],dparams=[0.34,0.1,1],mean='ARX', vol='GARCH',p=1,o=1,q=2,lags=[1,2],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.053361718759494026, 0.946638281240506]
Means:  [-0.4943344076887864, 0.027865483742811226]
Variances:  [7.024708410376458, 0.65956872083772]
Whether converge: True .   Optimization terminated successfully


D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bo


 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.9535589381552456, 0.04644106184475438]
Means:  [0.016566225560602842, -0.3401488214807695]
Variances:  [0.5912071849690014, 6.902251390977988]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.445
Mean Model:                                       AR-X   Adj. R-squared:                  0.434
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -162.994
Distribution:      Mixture of two Normal distributions   AIC:                           351.988
Method:                             Maximum Likelihood   BIC:                           395.251
                                                         No. Observations:                  206
Date:                                 Tue, Jul 25 2023   Df Residuals:                      201
Time:                                         20:16:27   Df Model:                            5
                                         Mean Model                                        
                                 coef    std

In [284]:
SPARCH(Y = sample_data['Inflation'],X=sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']],dparams=[0.34,-0.1,1],mean='ARX', vol='GARCH',p=2,o=2,q=2,lags=[1,2],cov = 'robust')

 Using residuals of t distribution to estimate mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.6547394940996469, 0.34526050590035307]
Means:  [0.7261654618252837, -1.3770738297109022]
Variances:  [1e-05, -2.596123404071845]
Whether converge: True .   Optimization terminated successfully


C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_10388/3611180394.py:8: RuntimeWarning: invalid value encountered in double_scalars
  lls =-np.log(volatility) +   np.log(p1 * norm.pdf(z, u1, sigma_1_2**0.5) + p2 * norm.pdf(z, u2, sigma_2_2**0.5))
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: Runti


 SPARCH estimation result: mixture of 2 normals parameters 
Mixture of 2 normals parameters: 
Probabilities:  [0.9267572599873658, 0.0732427400126342]
Means:  [0.028430383163487255, -0.3597361867734523]
Variances:  [0.5719931842817138, 4.648802593548418]


                                 AR-X - GJR-GARCH Model Results                                
Dep. Variable:                               Inflation   R-squared:                       0.440
Mean Model:                                       AR-X   Adj. R-squared:                  0.429
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -159.903
Distribution:      Mixture of two Normal distributions   AIC:                           349.807
Method:                             Maximum Likelihood   BIC:                           399.725
                                                         No. Observations:                  206
Date:                                 Tue, Jul 25 2023   Df Residuals:                      201
Time:                                         20:23:50   Df Model:                            5
                                         Mean Model                                        
                                 coef    std

### Simulation data generated from SPARCH 

In [136]:
def MofN_simulator(dist_params=[0.2,-0.6,0.7],size=5000):
    """
    Simulate SPARCH data with given distributional parameters
    
    Input:  dist_params[p1,u1,sigma1_2]
    return: DataFrame
        1.data: The simulated data, which includes any mean dynamics.
        2.volatility: The conditional volatility series
        3.errors: The simulated errors generated to produce the model. 
                    The errors are the difference between the data and its conditional mean, 
                    and can be transformed into the standardized errors by dividing by the volatility.
    """
    print('Simulating SPARCH data. Error follows: ')
    print_param(dist_params)
    gjr = arch_model(y=sample_data['Inflation shock'],mean='Zero', vol='GARCH',p=2, o=2,q=1).fit(disp='off')
    fake_params = list(gjr.params) + dist_params
    sim_sparch = arch_model(None, p=2, o=2, q=1, mean='Zero')
    sim_sparch.distribution = MixNormal()
    return sim_sparch.simulate(fake_params, size),fake_params
sim_data,fake_params = MofN_simulator()

Simulating SPARCH data. Error follows: 
Mixture of 2 normals parameters: 
Probabilities:  [0.2, 0.8]
Means:  [-0.6, 0.15]
Variances:  [0.7, 1.046875]


### Test whether MLE can estimate Mixture of 2 normals given conditional volatilities

Log likelihood function is
$$
 L\left(\epsilon_t\right) =  \sum_{t=1}^{T} \ln \frac{1}{\sqrt{h_t}} \left[
 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(z_t-\mu_1)^2}{2\sigma_1^2} \}
+p_2\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(z_t-\mu_2)^2}{2\sigma_2^2} \} \right]
$$,
where $z_t = \frac{\epsilon_t}{\sqrt{h_t}}$

In [294]:
sparch = arch_model(sim_data.data, p=2, o=2, q=1, mean='Zero')
sparch.distribution = MixNormal()
sp = sparch.fit()#starting_values = fake_params)
sp

Iteration:      1,   Func. Count:     11,   Neg. LLF: nan
Iteration:      2,   Func. Count:     22,   Neg. LLF: inf
Iteration:      3,   Func. Count:     33,   Neg. LLF: 5600.741477372661
Iteration:      4,   Func. Count:     46,   Neg. LLF: 5842.353019223569
Iteration:      5,   Func. Count:     57,   Neg. LLF: 8156.19207707535
Iteration:      6,   Func. Count:     68,   Neg. LLF: nan
Iteration:      7,   Func. Count:     79,   Neg. LLF: nan
Iteration:      8,   Func. Count:     92,   Neg. LLF: 4535.594198789753
Iteration:      9,   Func. Count:    103,   Neg. LLF: 5079.848267321051
Iteration:     10,   Func. Count:    114,   Neg. LLF: 4190.979471747992
Iteration:     11,   Func. Count:    125,   Neg. LLF: 4328.354357135434
Iteration:     12,   Func. Count:    136,   Neg. LLF: 4175.417205548612
Iteration:     13,   Func. Count:    147,   Neg. LLF: 4207.023509371622
Iteration:     14,   Func. Count:    158,   Neg. LLF: 4084.3742229443005
Iteration:     15,   Func. Count:    169,   Neg.

D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bo

Iteration:     30,   Func. Count:    324,   Neg. LLF: 4063.902648039336
Iteration:     31,   Func. Count:    334,   Neg. LLF: 4063.902184288549
Iteration:     32,   Func. Count:    344,   Neg. LLF: 4063.902085826351
Iteration:     33,   Func. Count:    354,   Neg. LLF: 4063.9020751992903
Iteration:     34,   Func. Count:    364,   Neg. LLF: 4063.902069537122
Iteration:     35,   Func. Count:    373,   Neg. LLF: 4063.902069537098
Optimization terminated successfully    (Exit mode 0)
            Current function value: 4063.902069537122
            Iterations: 35
            Function evaluations: 373
            Gradient evaluations: 35


                              Zero Mean - GJR-GARCH Model Results                              
Dep. Variable:                                    data   R-squared:                       0.000
Mean Model:                                  Zero Mean   Adj. R-squared:                  0.000
Vol Model:                                   GJR-GARCH   Log-Likelihood:               -4063.90
Distribution:      Mixture of two Normal distributions   AIC:                           8145.80
Method:                             Maximum Likelihood   BIC:                           8204.46
                                                         No. Observations:                 5000
Date:                                 Tue, Jul 25 2023   Df Residuals:                     5000
Time:                                         20:31:23   Df Model:                            0
                              Volatility Model                             
                 coef    std err          t      P>|t|     9

In [295]:
print_param(sp.params[-3:])

Mixture of 2 normals parameters: 
Probabilities:  [0.5046679889107604, 0.49533201108923963]
Means:  [0.15084710405835253, -0.15369025811745027]
Variances:  [1.1377469048024764, 0.811970260793873]


In [292]:
fake_params

[0.12425749794418403,
 0.24259235652414185,
 0.7767350969379752,
 -0.11478216666540393,
 -0.7767350969379752,
 0.11944293967026282,
 0.2,
 -0.6,
 0.7]

### Comments:

I simulated 1000 data points given parameter values. SPARCH yields similar parameters to true model. True parameters are within one std error of estimated parameters.
My conclusion is that my code and estimation is reliable.